# Dryden Turbulence Model


## Foreward

Notation:

|   type   | longitude | lateral |  vertical |
|----------|:---------:|:-------:|:---------:|
|linear    | u         |  v      | w         |
|angular   | p         |  r      | q         |


Begin with some setup, importing and defining constants



In [ ]:
# Imports
import math
import numpy as np
import scipy as sp
from scipy.signal import lti
from scipy.interpolate import interp2d
import matplotlib.pyplot as plt



In [ ]:
# Constants
M_TO_FT = 3.28084


## Given

Wingspan, b \[m\] : 3.96

Pose:

- Position\[m\] : $\left[ 150, 22, 457.2 \right]$

- Velocity\[$\frac{m}{s}$\] : $\left[ 1.2, 1.5, 0.12 \right]$




In [ ]:
# given
wingspan_m = 3.96

pose = {'position_m':[150, 22, 457.2]}
velocity_m_per_s = np.array([9.85, -0.72, 0.12])



## Assumption

The Dryden Model defines the filtering intensity to be a function of the turbulence at 20ft above launch. Moreover, it outlines a rough mapping to use with the following:

|Turbulence |  Wind Speed(kts) |
|----------:|:---------:|
|Light      | 15        |
|Moderate   | 30        |
|Severe     | 45        |

We will assume that the wind turbulence 20ft above launch to be Moderate,  that is 30 kts

In [ ]:
LAUNCH_TURBULENCE_INTENSITY_LIGHT_kts       = 15 
LAUNCH_TURBULENCE_INTENSITY_MODERATE_kts    = 30 
LAUNCH_TURBULENCE_INTENSITY_SEVER_kts       = 45 

launch_turbulence_intensity = LAUNCH_TURBULENCE_INTENSITY_MODERATE_kts

## Find

The:
- linear rate
- angular rate

of the wind turbulence over the time interval: $[0, t]$, with a sample period of $dt$ where:

$$ t = 1000 s$$
$$ dt = 0.01 s$$ 






In [ ]:
use_noise_from_file = True

# noise_file = 'data/noise_with_scale_uvw.csv'
noise_file = 'data/noise_without_scale_uvw.csv'

if use_noise_from_file:
    my_data = np.genfromtxt(noise_file, delimiter=',')
    
    t_s = my_data[:,0]
    noise_u = my_data[:,1]
    noise_v = my_data[:,2]
    noise_w = my_data[:,3]

    # still use randomly generated noise for angular components
    noise_intensity = 0
    noise_variance = 1
    noise_p = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_q = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_r = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])

    sim_start_s = t_s[0]
    sim_end_s = t_s[-1]
    dt_s = t_s[1] - t_s[0]

    noise_scale = np.sqrt(math.pi / dt_s)    

else:
    sim_start_s = 0
    sim_end_s = 1000
    dt_s = 0.01

    noise_intensity = 0
    noise_variance = 1
    noise_scale = np.sqrt(math.pi / dt_s)

    t_span = [sim_start_s, sim_end_s]
    t_s = np.arange(t_span[0], t_span[1] + dt_s/2, dt_s)
    noise_u = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_v = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_w = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_p = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_q = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])
    noise_r = np.random.normal(noise_intensity, np.sqrt(noise_variance), t_s.shape[0])



For validity, we can observe the noise is indeed gaussian from:

In [ ]:
# Time Trace
plt.title('Time Trace of Noise_u')
plt.plot(t_s, noise_u)
plt.xlabel('time(s)')
plt.ylabel('Intensity(ft/s)')




In [ ]:
# Histogram
plt.hist(noise_u,bins='auto')
plt.title('Histogram of Noise_u')
plt.xlabel('Intensity(ft/s)')
plt.ylabel('Count')

In [ ]:
from scipy.signal.windows import hann

fs = 1 / dt_s # sample rate, number of samples per unit of time

nblock = int(len(t_s) / 100) # 1024
overlap = int(nblock / 2)   # 512
win = hann(nblock, False)

print(nblock)
print(overlap)

SF = np.linalg.norm(win)**2 / np.sum(win)**2

In [ ]:

freq_noise_u, psd_noise_u = sp.signal.welch(noise_u * noise_scale, fs, window=win, noverlap=overlap, nfft=nblock, scaling='spectrum', detrend=False)

# PSD
plt.plot(freq_noise_u, psd_noise_u)

plt.title('Power Spectral Density of Noise_u')
plt.xlabel('$f$ (Hz)')
plt.ylabel('$\Phi_u$ ($ V^2/Hz $)')




## Analysis



The Dryden model defines turbulent wind to be a WSS stochasctic process such that:

$$ Y_i(f) = H_i(f) * X_i(f) $$

Where:

$i,\ Wind\ Component\ ([u,v,w,p,q,r])$

$X(f),\ Input:\ Random\ Noise$

$Y(f),\ Output:\ Turbulent\ Wind\ Rate$

$H(f),\ Transfer Function:\ Filter\ on\ Noise$


Examing Specifically the Linear-Longitudnal(u) component of Turbulence in the frequency domain:
$$ Y_u(s) = H_u(s) * X_u(s) $$

The Dryden model determined(from empirical data) the following to be the respective transfer functions for each component of the turbulence wind rate: 


Linear-Longitudanal, u
$$ H_u(s) = \sigma_u  \sqrt{\frac{2L_u}{\pi V}} \cdot  \frac{1}{1 + \frac{L_u}{V}s}$$
Linear-Lateral, v
$$ H_v(s) = \sigma_v  \sqrt{\frac{L_v}{\pi V}} \cdot \frac{1 + \sqrt{3}\frac{L_v}{V}s}{\left(1 + \frac{L_v}{V}s\right)^2}$$
Linear-Vertical, w
$$ H_w(s) = \sigma_w  \sqrt{\frac{L_w}{\pi V}} \cdot \frac{1 + \sqrt{3}\frac{L_w}{V}s}{\left(1 + \frac{L_w}{V}s\right)^2}$$
Angular-Longitudanal, p
$$H_p(s) = \sigma_w \sqrt{\frac{0.8}{V}}\cdot\frac{\left(\frac{\pi}{4b}\right)^\frac{1}{6})}{L_w^\frac{1}{3}\left(1 + \left(\frac{4b}{\pi V}\right)s\right)}$$
Angular-Longitudanal, q
$$H_r(s) = \frac{\mp \frac{1}{V}s}{1 + \left(\frac{3b}{\pi V}\right)s} \cdot H_v(s)$$
Angular-Longitudanal, r
$$H_q(s) = \frac{\pm \frac{1}{V}s}{1 + \left(\frac{4b}{\pi V}\right)s} \cdot H_w(s)$$

Where:

$V, aircraft\ velocity$

$b, aircraft\ wingspan$

$L_i, scale\ length$

$\sigma_i, intensity$

Where again the Dryden model defines scale length and intensity in terms of the wind rate component and a function of the aircraft altitude and a relative characterization of the wind speed at 20ft above launch conditions, $W_{20}$, such that:

#### Altitude: \[ < 1000ft\]

$$L_u = L_v = \frac{h}{\left(0.177 + 0.000823h\right)^{1.2}}$$
$$ L_w = h $$
$$ \sigma_w = 0.1W_{20} $$
$$ \sigma_u = \sigma_v = \frac{1}{(0.177 + 0.000823h)^{0.4}}$$

#### Altitude: \[1000ft, 2000ft\]

$$ L_{i} = interp(h, L_{i}(h=1000ft), L_{i}(h=2000ft))$$


<center>
    <img src='./img/MidHigh_Turbulence_Intensity.png' width="50%" height="50%">
</center>

#### Altitude: \[ > 2000ft \]
$$ L_u = L_v = L_w = 2500 $$
$$ \sigma_u = \sigma_v = \sigma_w $$ 


### Calculating Scale Length And Intensities



In [ ]:
# Probability of Exceedance Table
# Figure 7 from p. 49 of MIL-F-8785C


poe_altitude_ft = np.array([500.0, 1750.0, 3750.0, 7500.0, 15000.0, 25000.0, 35000.0, 45000.0, 55000.0, 65000.0, 75000.0, 80000.0])
poe_index = np.array([1,2,3,4,5,6,7]) # 3: Light, 4: Moderate, 6: Severe
poe_table = np.array([
    [ 3.2,   2.2,   1.5,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 4.2,   3.6,   3.3,   1.6,   0.0,   0.0,   0.0,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 6.6,   6.9,   7.4,   6.7,   4.6,   2.7,   0.4,   0.0,   0.0,   0.0,  0.0,  0.0],
    [ 8.6,   9.6,  10.6,  10.1,   8.0,   6.6,   5.0,   4.2,   2.7,   0.0,  0.0,  0.0],
    [11.8,  13.0,  16.0,  15.1,  11.6,   9.7,   8.1,   8.2,   7.9,   4.9,  3.2,  2.1],
    [15.6,  17.6,  23.0,  23.6,  22.1,  20.0,  16.0,  15.1,  12.1,   7.9,  6.2,  5.1],
    [18.7,  21.5,  28.4,  30.2,  30.7,  31.0,  25.2,  23.1,  17.5,  10.7,  8.4,  7.2]
])

poe_lookup = interp2d(poe_altitude_ft, poe_index, poe_table)


In [ ]:
altitude_m = pose['position_m'][2]
altitude_ft = altitude_m * M_TO_FT
wingspan_ft = wingspan_m * M_TO_FT 

# def get_scale_length_and_intensities(altitude_ft, wingspan_ft):

#     scale_length = dict('')
#     intensity = dict()

if altitude_ft < 1000:
    scale_length_longitude  = altitude_ft / (0.177 + 0.000823*altitude_ft)**1.2
    scale_length_lateral    = scale_length_longitude
    scale_length_vertical   = altitude_ft

    intensity_longitude = 0.1*launch_turbulence_intensity / (0.177 + 0.000823*altitude_ft)**0.4
    intensity_lateral   = intensity_longitude
    intensity_vertical  = 0.1*launch_turbulence_intensity

elif altitude_ft > 2000:
    scale_length_longitude  = 2500
    scale_length_lateral    = 2500
    scale_length_vertical   = 2500

    intensity_longitude     = poe_lookup(altitude_ft, 4.0)
    intensity_lateral       = intensity_longitude
    intensity_vertical      = intensity_longitude

else:
    low = 1000 / (0.177 + 0.000823*1000)**1.2
    high = 2500

    print('low', low)
    print('high', high)

    intensity_vert_low = 0.1*launch_turbulence_intensity
    intensity_long_low = 0.1*launch_turbulence_intensity / (0.177 + 0.000823*1000)**0.4

    intensity_high     = poe_lookup(2000.0, 4.0)[0]

    scale_length_longitude  = np.interp(altitude_ft, [1000.0, 2000.0], [low, high])
    scale_length_lateral    = np.interp(altitude_ft, [1000.0, 2000.0], [low, high])
    scale_length_vertical   = np.interp(altitude_ft, [1000.0, 2000.0], [1000.0, high])

    intensity_longitude     = np.interp(altitude_ft, [1000.0, 2000.0], [intensity_long_low, intensity_high])
    intensity_lateral       = np.interp(altitude_ft, [1000.0, 2000.0], [intensity_long_low, intensity_high])
    intensity_vertical      = np.interp(altitude_ft, [1000.0, 2000.0], [intensity_vert_low, intensity_high])



print('Scale Length, Longitude: %d' % scale_length_longitude)
print('Scale Length, Lateral: %d'   % scale_length_lateral)
print('Scale Length, Vertical: %d'  % scale_length_vertical)

print('Intensity, Longitude: %d' % intensity_longitude)
print('Intensity, Lateral: %d'   % intensity_lateral)
print('Intensity, Vertical: %d'  % intensity_vertical)


### Filters / Transfer Functions

For a WSS stochastic process:

$$\Phi_Y(f) = \left| H(f) \right|^2 \Phi_X(f)$$

$$ H(f) = \frac{Y(f)}{X(f)} $$

Using the PSD noted in the extra work section, transfer functions resolve to:

### Transfer Functions, Linear Rate


Linear-Longitudanal
$$ H_u(s) = \sigma_u  \sqrt{\frac{2L_u}{\pi V}} \cdot  \frac{1}{1 + \frac{L_u}{V}s}$$
Linear-Lateral
$$ H_v(s) = \sigma_v  \sqrt{\frac{L_v}{\pi V}} \cdot \frac{1 + \sqrt{3}\frac{L_v}{V}s}{\left(1 + \frac{L_v}{V}s\right)^2}$$
Linear-Vertical
$$ H_w(s) = \sigma_w  \sqrt{\frac{L_w}{\pi V}} \cdot \frac{1 + \sqrt{3}\frac{L_w}{V}s}{\left(1 + \frac{L_w}{V}s\right)^2}$$



In [ ]:
# linear rate filter
# given
# - sccale lengths
# - intensities
# - Velocity

velocity_ft_per_s = velocity_m_per_s * M_TO_FT
velocity = np.linalg.norm(velocity_ft_per_s)

velocity_scale_longitude    = scale_length_longitude / velocity
velocity_scale_lateral      = scale_length_lateral / velocity
velocity_scale_vertical     = scale_length_vertical / velocity

print(velocity_scale_longitude)
print(velocity_scale_lateral  )
print(velocity_scale_vertical )

# numerators denoted with 'b'
# denominators denoted with 'a'

b_u     = intensity_longitude * math.sqrt(2 * velocity_scale_longitude / math.pi)
b0_u    = b_u * 1
as_u    = velocity_scale_longitude
a0_u    = 1

b_v     = intensity_lateral * math.sqrt(velocity_scale_lateral / math.pi)
bs_v    = b_v * math.sqrt(3.0) * velocity_scale_lateral
b0_v    = b_v
as2_v   = velocity_scale_lateral ** 2
as_v    = 2 * velocity_scale_lateral
a0_v    = 1

b_w     = intensity_vertical * math.sqrt(velocity_scale_vertical / math.pi)
bs_w    = b_w * math.sqrt(3.0) * velocity_scale_vertical
b0_w    = b_w
as2_w   = velocity_scale_vertical ** 2
as_w    = 2 * velocity_scale_vertical
a0_w    = 1

H_u = lti(  [b0_u],
            [as_u, a0_u])

H_v = lti( [bs_v, b0_v], 
           [as2_v, as_v, a0_v])

H_w = lti( [bs_w, b0_w], 
           [as2_w, as_w, a0_w])

print(H_u)
print(H_v)
print(H_w)


### Transfer Functions, Angular Rate

Angular-Longitudanal
$$H_p(s) = \sigma_w \sqrt{\frac{0.8}{V}}\cdot\frac{\left(\frac{\pi}{4b}\right)^\frac{1}{6})}{L_w^\frac{1}{3}\left(1 + \left(\frac{4b}{\pi V}\right)s\right)}$$

Angular-Longitudanal
$$H_r(s) = \frac{\mp \frac{1}{V}s}{1 + \left(\frac{3b}{\pi V}\right)s} \cdot H_v(s)$$

Angular-Longitudanal
$$H_q(s) = \frac{\pm \frac{1}{V}s}{1 + \left(\frac{4b}{\pi V}\right)s} \cdot H_w(s)$$

In [ ]:
# angular rate filter
k =  wingspan_ft / (math.pi * velocity)

b0_p    = intensity_vertical * math.sqrt(0.8 / velocity) * ((math.pi / (4 * wingspan_ft)) ** (1 / 6)) / ((scale_length_vertical) ** (1 / 3))
as_p    = 4 * k
a0_p    = 1

bs2_q   = -b_w * math.sqrt(3.0) * velocity_scale_vertical / velocity
bs_q    = -b_w / velocity
b0_q    = 0
as3_q   = 3 * k * velocity_scale_vertical ** 2
as2_q   = velocity_scale_vertical ** 2 + 2 * 4 * k * velocity_scale_vertical
as_q    = 4 * k + 2 * velocity_scale_vertical
a0_q    = 1

bs2_r   = b_v * math.sqrt(3.0) * velocity_scale_lateral / velocity
bs_r    = b_v / velocity
b0_r    = 0
as3_r   = 3 * k * velocity_scale_lateral ** 2
as2_r   = velocity_scale_lateral ** 2 + 2 * 3 * k * velocity_scale_lateral
as_r    = 3 * k + 2 * velocity_scale_lateral
a0_r    = 1


H_p = lti(  [b0_p],
            [as_p, a0_p])

H_q = lti( [bs_q, b0_q], 
           [as3_q, as2_q, as_q, a0_q])

H_r = lti( [bs2_r, bs_r, b0_r], 
           [as3_r, as2_r, as_r, a0_r])

print(H_p)
print(H_q)
print(H_r)

### Filter Noise 
 We then filter the discrete noise samples via the appropriate transfer function, solving:
 $$ Y_i(f) = H_i(f) * X_i(f) $$

In [ ]:
turbulence_linear_rate = np.array([
                                    H_u.output(noise_scale * noise_u, t_s)[1],
                                    H_v.output(noise_scale * noise_v, t_s)[1],
                                    H_w.output(noise_scale * noise_w, t_s)[1]])

turbulence_angular_rate = np.array([
                                    H_p.output(noise_p,	t_s)[1],
                                    H_q.output(noise_q,	t_s)[1],
                                    H_r.output(noise_r,	t_s)[1]])



plt.plot(t_s, turbulence_linear_rate[0])
plt.ylabel('linear rate - longitude (ft/s)')
plt.xlabel('time(s)')


In [ ]:
if t_s.size > 1:
    plt.plot(t_s, turbulence_linear_rate[1])
    plt.ylabel('linear rate - lateral (ft/s)')
    plt.xlabel('time(s)')

In [ ]:
if t_s.size > 1:
    plt.plot(t_s, turbulence_linear_rate[2])
    plt.ylabel('linear rate - vertical (ft/s)')
    plt.xlabel('time(s)')
    

At this point its useful to make a quick observation on the effects of converting the output to the metric system and how that would change the base transfer functions if the input was also in metric. 

$$  $$

# Extra Work 

While the transfer function solution is valid, it is presented here as a convenience as the implementation of this model will be in C++ where the convenient scipy libraries of python do not exist. While it may be possible to use math libraries we will go into further methods for solving the same set of equations here as a practice.

The two methods observed here are:
- Discretized Difference-equations
- Discretized State-Space Model


### Discretized Difference Equations

The Dryden paper does present a set of equations that give the solution at a discrete sample time of $\Delta T$

$$u_g = \left(1 - \frac{V}{L_u}\Delta T\right)u_{g-prev} + \sigma_u \sqrt{2 \frac{V}{L_u}\Delta T} \frac{\eta_1}{\sigma_\eta}$$
$$v_g = \left(1 - \frac{V}{L_u}\Delta T\right)v_{g-prev} + \sigma_v \sqrt{2 \frac{V}{L_u}\Delta T} \frac{\eta_2}{\sigma_\eta}$$
$$w_g = \left(1 - \frac{V}{L_u}\Delta T\right)w_{g-prev} + \sigma_w \sqrt{2 \frac{V}{L_u}\Delta T} \frac{\eta_3}{\sigma_\eta}$$

$$p_g = \left( 1 - \frac{2.6}{\sqrt{L_w b }\Delta T} \right)p_g + \left( \sqrt{2\frac{2.6}{\sqrt{L_w b}}\Delta T} \right)\left( \frac{0.95}{\frac{\sqrt[3]{2 L_w b^2}}{\sigma_\eta}\eta_4} \right) \sigma_w$$
$$r_g = \left( 1 - \frac{\pi V}{3 b} \right) r_g \mp \frac{\pi}{3b}(v_g - v_{g-1})$$
$$q_g = \left( 1 - \frac{\pi V}{4 b} \right) q_g \mp \frac{\pi}{4b}(w_g - w_{g-1})$$

where:

$$ \eta_i =random\ gaussian\ input\ for\ step$$



In [ ]:
## Stepping through discretized model:

linear_rate_u = [0]
linear_rate_v = [0] 
linear_rate_w = [0]

a_u = (1 - velocity/scale_length_longitude * dt_s)
b_u = intensity_longitude * math.sqrt(2 * velocity / scale_length_longitude * dt_s)

a_v = (1  - 2 * velocity/scale_length_lateral * dt_s)
b_v = intensity_lateral * math.sqrt(4 * velocity / scale_length_lateral * dt_s)

a_w = (1  - 2 * velocity/scale_length_vertical * dt_s)
b_w = intensity_vertical * math.sqrt(4 * velocity / scale_length_vertical * dt_s)

for step in range(1,len(t_s)):
    rate_u =  linear_rate_u[step - 1] *  a_u \
                + noise_u[step] * b_u

    rate_v =  linear_rate_v[step - 1] *  a_v \
                + noise_v[step] * b_v

    rate_w =  linear_rate_w[step - 1] *  a_w \
                + noise_w[step] * b_w


    linear_rate_u.append(rate_u)
    linear_rate_v.append(rate_v)
    linear_rate_w.append(rate_w)

plt.plot(t_s, linear_rate_u)
plt.ylabel('linear rate - longitude (ft/s)')
plt.xlabel('time(s)')



In [ ]:
plt.plot(t_s, linear_rate_v)
plt.ylabel('linear rate - lateral (ft/s)')
plt.xlabel('time(s)')

In [ ]:
plt.plot(t_s, linear_rate_w)
plt.ylabel('linear rate - vertical (ft/s)')
plt.xlabel('time(s)')

### State Space Solution From Transfer Function

For the instance that such quations werent conveniently provided, we can instead determine them from the Discrete State Space Equations, which derive from the Continuous State Space Equations, that is we will determine:

$$ H(s) -> SS_{Continuous} -> SS_{Discrete}$$

For convenience, the Discrete State Space will be abbreviated: $SS_D$

The process for determing the State Space from a transfer function is a trivial one, especially for our set of functions where the order of the numerator is always less than the denominator, see Reference 3.

From the continuous state space solution:

$$\pmb{\dot{x} = A x + B u}$$
$$\pmb{y = C x + D u}$$

where $\pmb{y}$ is the objective.

At this point traditional numerical solutions to the set of ordinary differential equations can be used like the Euler Method, Heun Method, or Runga Kutta. 

However for a simulation that runs at relatively set time steps, we can leverage the ability to discretize the continuous model on the time step as the sampling period. This is done from the known conversion that :

$$ A_D = e^{AT} = \Phi(T) $$
$$ B_D  = (A_D - I) A^{-1}B$$
$$ C_D = C$$
$$ D_D = D$$

where
$$ T =\ sample\ period$$

this results in the solution that:

$$x[k+1] = A_D x[k] + B_D U[k]$$
$$y[k+1] = C_D x[k] + D_D U[k]$$

where
$$ U = System Input $$




Examing the case of the transfer functions found before, we find:

In [ ]:
### Converting TF to SS

SS_u = H_u.to_ss()
SS_v = H_v.to_ss()
SS_w = H_w.to_ss()
SS_p = H_p.to_ss()
SS_q = H_q.to_ss()
SS_r = H_r.to_ss()

print(H_u)
print(SS_u)
print(SS_v)
print(SS_w)


Converting these into their discretized form with the sampling period being the same as that provided to the transfer functions:

In [ ]:
# Discrete SS from Continuous SS, longitudanal case

A = SS_u.A
B = SS_u.B
C = SS_u.C
D = SS_u.D

A_D = sp.linalg.expm(A * dt_s)
B_D = (A_D - sp.sparse.identity(A_D.shape[0])) * sp.linalg.inv(A) * B
C_D = C
D_D = D

print(A_D)
print(B_D)
print(C_D)
print(D_D)

In [ ]:

linear_rate_u = [0]

x_prev = 0

for step in range(1,len(t_s)):
    
    x = A_D * x_prev + B_D * noise_scale * noise_u[step - 1]
    y = C_D * x_prev + D_D * noise_scale  * noise_u[step - 1]

    x_prev = x

    linear_rate_u.append(y)


plt.plot(t_s, linear_rate_u)
plt.ylabel('linear rate - lateral (ft/s)')
plt.xlabel('time(s)')

Configrming Discrete State Space Solution matches that of built in Libraries

In [ ]:
ssD_u = SS_u.to_discrete(dt_s)

print(ssD_u)

linear_rate_u = ssD_u.output(noise_u * noise_scale ,t_s)[1]

if len(linear_rate_u) < len(t_s):
    linear_rate_u = np.concatenate(([np.array([0]),linear_rate_u[:,0]]))

plt.plot(t_s, linear_rate_u)
plt.ylabel('linear rate - longitude (ft/s)')
plt.xlabel('time(s)')

In [ ]:
ssD_v = SS_v.to_discrete(dt_s)

print(ssD_v)

linear_rate_v = ssD_v.output(noise_v * noise_scale ,t_s)[1]

if len(linear_rate_v) < len(t_s):
    linear_rate_v = np.concatenate(([np.array([0]),linear_rate_v[:,0]]))

plt.plot(t_s, linear_rate_v)
plt.ylabel('linear rate - lateral(ft/s)')
plt.xlabel('time(s)')

In [ ]:
ssD_w = SS_w.to_discrete(dt_s)

print(ssD_w)

linear_rate_w = ssD_w.output(noise_w * noise_scale ,t_s)[1]

if len(linear_rate_w) < len(t_s):
    linear_rate_w = np.concatenate(([np.array([0]),linear_rate_w[:,0]]))

plt.plot(t_s, linear_rate_w)
plt.ylabel('linear rate - vertical (ft/s)')
plt.xlabel('time(s)')

## Analysis Of Results

Lastly it would be good to confirm that the ascertained results are a valid representation of the Dryden Model. 

In theory a spectral density estimation of the output should match that of Power Spectral Density equations that define the transfer function.



### Power Spectral Densities (Theoretical)

The derived equations for the discritized turbulent wind stem from a set of power spectral densities; therefore, the estimated turbulence rates should provide an estimated power spectral density that matches closely against the theoretical power spectral density as obtained from the source equations.


$$ \Phi_u(\omega) = \frac{2 \sigma^2 L_u}{\pi V} \cdot \frac{1}{1 + \left(\frac{L_u}{V}\omega\right)^2} $$

In [ ]:
Npts = 200
freq_theo = np.logspace(-2, 2, Npts)
w = 2.0 * math.pi * freq_theo

In [ ]:
# Power Spectral Density, Dryden Theory
Ku = intensity_longitude**2 * velocity_scale_longitude / math.pi
psd_theo_u = Ku / (1 + (velocity_scale_longitude * w)**2)
psd_theo_u = 10 * np.log10(psd_theo_u)


plt.semilogx(freq_theo, psd_theo_u)
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])

### Power Spectral Densities (Expiremental)
#### DTFT/FFT

Signal analysis tells us that for a simulated data signal, like that of the output from the discretized solution, will have a power spectral

In [ ]:
from scipy.signal.windows import hann

freq_sample = 1 / dt_s # sample rate, number of samples per unit of time

nblock = int(len(t_s) / 100) # 1024
overlap = int(nblock / 2)   # 512
win = hann(nblock, False)

SF = np.linalg.norm(win)**2 / np.sum(win)**2

In [ ]:

freq_sim_u, psd_sim_u = sp.signal.welch(turbulence_linear_rate[0], freq_sample, window=win, noverlap=overlap, nfft=nblock, scaling='spectrum', detrend=False)
psd_sim_u = 10*np.log10(psd_sim_u)

plt.semilogx(freq_sim_u, psd_sim_u, '.', label='Simulated')
plt.semilogx(freq_theo, psd_theo_u, label='Theoretical Dryden')
plt.legend()
plt.grid()
plt.xlabel('frequency (Hz)')
plt.ylabel('PSD (dB)')
plt.xlim([0.01, 100])
plt.ylim([-80, 20])




Notes:

For a time signal:

Power Spectrum - which frequences contian the signal's power. Measure of the distribution of power(average of signal energy) values as a function of frequency. The square of the FFT's magnitude. Area under the signal plot useing discrete fourier transform

Power Spectral Density - Measure of signal's power content vs frequency. Assigns unit of power to each unit frequency



## Metric Representation

Its trivial to show then the metric form of the equations becomes:

#### Altitude: \[ < 304.8 m\]

$$L_u = L_v = \frac{h}{\left(0.177 + 0.0027h\right)^{1.2}}$$
$$ L_w = h $$
$$ \sigma_w = 0.03048W_{20} $$
$$ \sigma_u = \sigma_v = \frac{1}{(0.177 + 0.0027h)^{0.4}}$$

#### Altitude: \[304.8m, 609.6m\]

$$ L_{i} = interp(h, L_{i}(h=304.8m), L_{i}(h=609.6m))$$

#### Altitude: \[ > 609.6m \]
$$ L_u = L_v = L_w = 762 $$
$$ \sigma_u = \sigma_v = \sigma_w $$ 

With a new mapping for launch turbulence intensity as:

|Turbulence |  Wind Speed(m/s) |
|----------:|:---------:|
|Light      | 7.71667   |
|Moderate   | 15.4333   |
|Severe     | 23.15     |

The tabulated form of the probability of exceedance lookup then becomes:

|  | 152.4 | 533.4 | 1143 | 2286 | 4572 | 7620 | 10668 | 13716 | 16764 | 19812 | 22860 | 24384 |
|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|:-:|
| 1 | 0.975 | 0.671 | 0.457 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 2 | 1.28 | 1.097 | 1.006 | 0.488 | 0 | 0 | 0 | 0 | 0 | 0 | 0 | 0 |
| 3 | 2.012 | 2.103 | 2.256 | 2.042 | 1.402 | 0.823 | 0.122 | 0 | 0 | 0 | 0 | 0 |
| 4 | 2.621 | 2.926 | 3.231 | 3.078 | 2.438 | 2.012 | 1.524 | 1.28 | 0.823 | 0 | 0 | 0 |
| 5 | 3.597 | 3.962 | 4.877 | 4.602 | 3.536 | 2.957 | 2.469 | 2.499 | 2.408 | 1.494 | 0.975 | 0.64 |
| 6 | 4.755 | 5.364 | 7.01 | 7.193 | 6.736 | 6.096 | 4.877 | 4.602 | 3.688 | 2.408 | 1.89 | 1.554 |
| 7 | 5.7 | 6.553 | 8.656 | 9.205 | 9.357 | 9.449 | 7.681 | 7.041 | 5.334 | 3.261 | 2.56 | 2.195 |

All subsequent equations related to the turbulence linear rate remain unchanged



# References

1. [Mathworks-Dryden(Continuous)][https://www.mathworks.com/help/aeroblks/drydenwindturbulencemodelcontinuous.html]

2. [Solving Stochastic Process with Markov Chain][https://apps.dtic.mil/sti/pdfs/ADA142055.pdf]

3. [Converting Transfer Function to State Space][https://ocw.mit.edu/courses/aeronautics-and-astronautics/16-30-feedback-control-systems-fall-2010/lecture-notes/MIT16_30F10_lec06.pdf]

4. [Application Turbulence Simulation Techniques with Application to FLight Analysis][https://ntrs.nasa.gov/api/citations/19800023518/downloads/19800023518.pdf]

5. [Transfer Function Relationship][https://dsp.stackexchange.com/questions/270/what-is-the-relation-between-the-psds-of-filter-input-and-output-called-r-y]